# 📗 손실함수와 경사하강: PyTorch로 직선 하나 학습시키기

딥러닝 모델은 **예측, 손실 계산, 기울기 계산, 파라미터 갱신**을 반복하며 학습합니다. 이 교안은 파라미터가 두 개(`w`, `b`)인 직선 모델 하나로 이 반복을 PyTorch로 직접 돌려 봅니다. 여기서 쓰는 `loss.backward()`, `optimizer.step()`은 교안 02의 신경망과 LLM 학습에서도 그대로 쓰입니다.

<img src="images/20_lesson01_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: PyTorch로 직선 하나 학습시키기'이고 그 아래 띠에 '학생 100명의 공부시간과 시험 점수'가 적혀 있습니다. 2행 2열의 판 네 개가 화살표로 이어집니다. 1 손실함수(MSE · MAE) 판에는 점과 직선 사이의 빨간 오차 막대가, 2 기울기(autograd) 판에는 'loss.backward()'에서 'w.grad'로 가는 화살표가, 3 경사하강 판에는 갱신 0·1·3·10·100번째 직선이 점들의 가운데로 모여드는 모습이, 4 학습률 판에는 느림·수렴·발산 손실 곡선 세 개가 그려져 있습니다.">

**학습 목표** (그림의 번호와 같습니다)

1. **MSE·MAE로 예측 오차를 계산하고, 이상치 한 명에 두 손실이 반응하는 차이를 설명합니다.**
2. **autograd로 기울기를 구해 숫자로 확인하고, 체인룰로 역전파가 하는 일을 설명합니다.**
3. **경사하강을 손으로 한 번 갱신한 뒤 `torch.optim.SGD`와 `nn.Linear`로 같은 학습을 만듭니다.**
4. **학습률에 따른 느림·수렴·발산을 비교하고, 입력 표준화와 학습률을 고르는 순서를 설명합니다.**

## 1. 모델이 학습한다는 것

학습 코드를 쓰기 전에, 학습이 무엇을 정하는 과정인지부터 정리합니다.

**예측 함수와 파라미터**
- **예측 함수:** 입력을 예측으로 바꾸는 규칙입니다. 이 교안은 직선 `점수 예측 = w × 공부시간 + b`를 씁니다.
- **파라미터(parameter):** 예측 함수 안에서 학습으로 정하는 숫자입니다. 직선에서는 가중치 `w`(공부시간에 곱하는 수)와 절편 `b`(공부시간이 0일 때의 예측) 두 개입니다.
- **학습:** 사람이 `w`와 `b`를 정하는 대신 데이터로 정하게 하는 일입니다. 아래 네 단계를 반복합니다.

<img src="images/21_training_loop.png" width="1050" alt="제목은 '학습: 네 단계를 반복해 손실을 줄입니다'입니다. 가운데 '반복' 원을 둘러싸고 네 카드가 시계 방향으로 이어집니다. 1 예측(점수 예측 = w × 공부시간 + b, 코드 predict(x, w, b)), 2 손실 계산(예측과 실제 점수가 얼마나 다른가, 코드 loss_fn(예측, y)), 3 기울기 계산(w와 b를 어느 쪽으로 바꾸면 손실이 늘어나나, 코드 loss.backward()), 4 파라미터 갱신(기울기 반대 방향으로 조금, 코드 optimizer.step()) 순서입니다. 아래 띠에는 '반복할수록 손실이 줄고 직선이 점들의 가운데로 옵니다'가 적혀 있습니다.">

5.4시간 공부하고 62점을 받은 학생 한 명으로 네 단계를 따라가 봅니다.

1. **예측:** `w=0`, `b=0`에서 시작하면 점수 예측은 0점입니다.
2. **손실 계산:** 실제 62점과 62점 차이가 납니다. 이 차이를 숫자 하나로 만든 것이 손실입니다.
3. **기울기 계산:** `w`와 `b`를 어느 쪽으로 바꾸면 차이가 커지는지 구합니다. 여기서는 줄이면 커지므로 늘려야 합니다.
4. **갱신:** `w`와 `b`를 늘리는 쪽으로 조금 바꿉니다. 학생 100명 전체로 이 반복을 100번 하면 공부 1시간당 약 6점, 공부시간 0일 때 약 36점인 직선이 됩니다. 이 교안에서 직접 확인합니다.

## 2. 실습 준비

학습에 넣을 데이터를 PyTorch가 계산할 수 있는 모양으로 준비합니다.

### 2-1. 라이브러리와 데이터

데이터를 읽는 pandas, 계산하는 PyTorch(`torch`), 그래프를 그리는 matplotlib을 불러오고 계산 장치를 정합니다. GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 학생 100명의 공부시간(시간)과 시험 점수
study = pd.read_csv(data_dir / "본실습" / "study_scores.csv")
print("행 수:", len(study))
display(study.head())

In [ ]:
# 공부시간이 길수록 점수가 높아지는지 산점도로 봅니다
plt.figure(figsize=(6, 4))
plt.scatter(study["공부시간"], study["점수"], s=15, alpha=0.6)
plt.xlabel("공부시간(시간)")
plt.ylabel("점수")
plt.title("공부시간과 시험 점수")
plt.show()

산점도의 점들이 오른쪽 위로 거의 직선을 이루며 늘어섭니다. 그래서 예측 함수를 직선 하나로 시작합니다.

### 2-2. 텐서 만들기

PyTorch는 텐서로 계산하므로 표의 열을 텐서로 바꿔야 합니다.

**텐서(tensor)**
- GPU에 올려 계산할 수 있는 배열입니다. 기울기를 자동으로 구하는 기능도 텐서에 붙어 있습니다.
- **`torch.tensor(값, dtype=torch.float32, device=device)`:** 값을 float32 텐서로 만들어 계산 장치에 올립니다. PyTorch 모델은 float32로 계산하는데 pandas의 숫자 열은 float64·int64라서 자료형을 맞춥니다.
- **모양 `(100, 1)`:** 학생 100명 × 입력 1개입니다. 뒤에서 쓰는 `nn.Linear`가 (데이터 수, 입력 수) 모양을 받으므로 처음부터 이 모양으로 만듭니다.
- **`.item()`:** 숫자 하나가 든 텐서에서 파이썬 숫자를 꺼냅니다. `round()`나 표에 넣을 때 씁니다.
- **`.cpu()`:** GPU에 있는 텐서를 CPU로 복사합니다. matplotlib은 CPU의 값만 그리므로 그래프 앞에서 씁니다.

In [ ]:
# 열 이름을 대괄호 두 겹으로 감싸면 100행 1열 표 모양 그대로 꺼내므로 텐서 모양이 (100, 1)이 됩니다
x_raw = torch.tensor(study[["공부시간"]].to_numpy(), dtype=torch.float32, device=device)
y = torch.tensor(study[["점수"]].to_numpy(), dtype=torch.float32, device=device)
print("모양:", x_raw.shape, "| 자료형:", x_raw.dtype, "| 장치:", x_raw.device)

# 앞 다섯 명의 공부시간
display(x_raw[:5])

### 2-3. 입력 표준화

입력의 크기가 학습률을 고르는 데 영향을 주므로 공부시간을 표준화해서 씁니다(표준화하지 않은 입력과는 학습률 절에서 비교합니다).

- **표준화(standardization):** 평균을 빼고 표준편차로 나눠 평균 0, 표준편차 1로 바꾸는 변환입니다. 군집 단원에서 StandardScaler로 한 것과 같은 변환입니다.
- 말로 하면 "평균보다 몇 시간 더(덜) 공부했는지를, 표준편차 몇 개만큼인지로 다시 센 값"입니다.
- **이름:** 원래 단위의 공부시간은 `x_raw`, 표준화한 공부시간은 `x`로 부릅니다. 점수 `y`는 점수 단위로 읽도록 그대로 둡니다.

<img src="images/24_standardization.png" width="1050" alt="제목은 '표준화: 평균에서 몇 표준편차 떨어졌나'입니다. 위 수직선 '공부시간(시간)'에 1.0, 5.4, 9.4 세 점과 '평균 5.08' 눈금이 있고, 아래 수직선 '표준화한 공부시간'에 같은 세 학생이 -1.47, 0.11, 1.55로 놓여 있으며 평균 자리는 0입니다. 위아래 점은 화살표로 이어지고, 가운데 상자에 '(공부시간 - 평균 5.08) ÷ 표준편차 2.78'이 적혀 있습니다.">

표준화로 세 학생의 공부시간이 바뀌는 과정입니다(공부시간 평균 5.08, 표준편차 2.78).

1. **1.0시간:** 평균보다 4.08시간 적으므로 (1.0 - 5.08) ÷ 2.78 = -1.47입니다.
2. **5.4시간:** 평균보다 0.32시간 많으므로 0.11로, 평균에 가깝습니다.
3. **9.4시간:** 평균보다 4.32시간 많으므로 1.55입니다.

In [ ]:
# 표준화한 공부시간: 0이면 평균만큼, 1이면 평균보다 표준편차 하나만큼 더 공부한 학생
x = (x_raw - x_raw.mean()) / x_raw.std()
print("공부시간 평균:", round(x_raw.mean().item(), 2), "| 표준편차:", round(x_raw.std().item(), 2))

# 아주 작은 음수가 -0.0으로 찍히지 않게 평균은 절댓값으로 봅니다
print("표준화한 공부시간 평균:", abs(round(x.mean().item(), 2)), "| 표준편차:", round(x.std().item(), 2))

# 앞 다섯 명의 표준화한 공부시간(1.0, 5.4, 4.1, 2.7, 9.4시간)
display(x[:5])

**결과 읽기**
- **공부시간 평균 5.08, 표준편차 2.78:** 표준화한 공부시간 1은 평균보다 2.78시간 더 공부한 학생입니다.
- **표준화한 공부시간 평균 0.0, 표준편차 1.0:** 표준화가 맞게 되었습니다.
- **앞 다섯 명:** 1.0시간은 -1.4679, 5.4시간은 0.1144, 9.4시간은 1.5528로 위 예시와 같습니다.

### 2-4. 예측 함수

손실을 계산하려면 먼저 파라미터로 예측을 만드는 함수가 있어야 합니다.

- `predict(x, w, b)`는 표준화한 공부시간 `x`와 파라미터 `w`, `b`를 받아 점수 예측 `w * x + b`를 돌려줍니다.
- 아직 학습 전이므로 `w=5`, `b=50`을 임의로 넣어 봅니다.

In [ ]:
def predict(x, w, b):
    """직선 w * x + b 로 점수 예측을 돌려줍니다."""
    return w * x + b


# 아직 학습 전이므로 임의로 고른 w=5, b=50으로 100명을 예측합니다
y_pred = predict(x, 5.0, 50.0)

# 앞 다섯 명의 표에 점수 예측 열을 붙여 나란히 봅니다
preview_table = study.head(5).copy()
preview_table["점수 예측"] = y_pred[:5].flatten().tolist()  # (5, 1) 모양을 한 줄로 펴서 열로 넣습니다
display(preview_table.round(2))

**결과 읽기**
- **첫째 학생(1.0시간):** 실제 46점, 예측 42.66점입니다.
- **다섯째 학생(9.4시간):** 실제 92점, 예측 57.76점으로 34점 넘게 적게 예측합니다. `w=5`, `b=50`이 둘 다 작아서, 많이 공부한 학생일수록 실제보다 낮게 예측합니다.
- 100명 각각의 이런 차이를 숫자 하나로 모은 것이 다음 절의 손실입니다.

## 3. 손실함수

어느 파라미터가 더 나은지 비교하려면 예측이 틀린 정도를 숫자 하나로 계산해야 합니다.

### 3-1. MSE와 MAE

숫자를 예측하는 문제(회귀)에서 많이 쓰는 두 손실의 차이를 알아야 데이터에 맞는 손실을 고를 수 있습니다.

**손실(loss)과 손실함수(loss function)**
- 손실은 예측이 실제와 다른 정도를 나타내는 숫자 하나이고, 손실을 계산하는 함수가 손실함수입니다. 손실이 작을수록 예측이 실제에 가깝습니다.
- **벌점:** 학생 한 명의 오차에 매기는 값입니다. MSE는 오차의 제곱, MAE는 오차의 절댓값을 벌점으로 쓰고, 두 손실은 벌점을 평균한 값입니다.
- **MSE(mean squared error, 평균제곱오차):** 오차(예측 - 실제)를 제곱해 평균합니다. 제곱하면 +오차와 -오차가 서로 지워지지 않고, 오차가 클수록 벌점이 제곱으로 커집니다.
- **MAE(mean absolute error, 평균절대오차):** 오차의 절댓값을 평균합니다. 벌점이 오차에 비례해 커집니다.

<img src="images/02_mse_mae.png" width="1050" alt="제목은 '오차 하나에 매기는 벌점: MSE와 MAE'입니다. 같은 세로 눈금을 쓰는 두 판에 막대가 값에 비례해 서 있습니다. 왼쪽 '오차 2' 판의 제곱(MSE) 막대 4와 절댓값(MAE) 막대 2는 바닥 가까이 낮습니다. 오른쪽 '오차 10' 판에서는 제곱(MSE) 막대 100이 판 꼭대기까지 닿고 절댓값(MAE) 막대 10은 그 10분의 1입니다. 아래 결론 상자에는 '오차가 5배 커지면 MSE 벌점은 25배, MAE 벌점은 5배'라고 적혀 있습니다.">

앞의 직선(`w=5`, `b=50`)에 벌점을 매기는 과정입니다.

1. **첫째 학생:** 예측 42.66점 - 실제 46점 = 오차 -3.34점입니다. MSE 벌점은 (-3.34)² = 11.16, MAE 벌점은 3.34입니다.
2. **다섯째 학생:** 예측 57.76점 - 실제 92점 = 오차 -34.24점입니다. MSE 벌점은 (-34.24)² = 1172.38, MAE 벌점은 34.24입니다.
3. **100명 평균:** 학생마다 구한 벌점을 평균하면 MSE 421.41, MAE 17.36입니다.

PyTorch는 두 손실을 `nn.MSELoss`, `nn.L1Loss`로 제공합니다(`torch.nn`은 신경망 부품을 모아 둔 모듈입니다). 먼저 정의대로 직접 계산한 값과 같은지 확인합니다.

In [ ]:
import torch.nn as nn

# 손실함수 객체: (예측, 정답)을 넣으면 평균한 손실 하나를 텐서로 돌려줍니다
loss_fn = nn.MSELoss()
mae_fn = nn.L1Loss()

# 정의대로 계산한 값과 PyTorch 손실함수의 값을 비교합니다
y_pred = predict(x, 5.0, 50.0)
print("정의대로 계산한 MSE:", round(((y_pred - y) ** 2).mean().item(), 2), "| nn.MSELoss:", round(loss_fn(y_pred, y).item(), 2))
print("정의대로 계산한 MAE:", round((y_pred - y).abs().mean().item(), 2), "| nn.L1Loss:", round(mae_fn(y_pred, y).item(), 2))

두 값이 같으므로 이제부터 `loss_fn`(MSE)과 `mae_fn`(MAE)을 씁니다. 다른 직선과 손실을 비교해 봅니다.

In [ ]:
# 임의로 고른 직선과 점들의 가운데를 지나도록 고른 직선의 손실을 비교합니다
loss_rows = []
for w_value, b_value in [(5.0, 50.0), (16.0, 66.0)]:
    y_pred = predict(x, w_value, b_value)
    loss_rows.append({"w": w_value, "b": b_value, "MSE": loss_fn(y_pred, y).item(), "MAE": mae_fn(y_pred, y).item()})
loss_table = pd.DataFrame(loss_rows)
display(loss_table.round(2))

In [ ]:
# 직선을 그릴 공부시간 0~10시간 50개를 (50, 1) 모양으로 만들고, 학생 데이터와 같은 평균, 표준편차로 표준화합니다
hours_grid = torch.linspace(0, 10, 50, device=device).reshape(50, 1)
x_grid = (hours_grid - x_raw.mean()) / x_raw.std()

# 두 직선을 산점도에 겹쳐 그립니다(가로축은 원래 공부시간)
plt.figure(figsize=(6, 4))
plt.scatter(x_raw.cpu(), y.cpu(), s=15, alpha=0.5, label="실제")
plt.plot(hours_grid.cpu(), predict(x_grid, 5.0, 50.0).cpu(), color="gray", label="w=5, b=50")
plt.plot(hours_grid.cpu(), predict(x_grid, 16.0, 66.0).cpu(), color="red", label="w=16, b=66")
plt.xlabel("공부시간(시간)")
plt.ylabel("점수")
plt.title("두 직선 비교")
plt.legend()
plt.show()

**표와 그래프 읽기**
- **첫 행(w=5, b=50):** MAE 17.36이면 학생 한 명당 평균 17점쯤 틀리게 예측합니다. 그래프에서 회색 직선이 오른쪽(많이 공부한 학생)으로 갈수록 점들보다 크게 아래에 있습니다.
- **둘째 행(w=16, b=66):** MSE 23.58, MAE 4.08로 두 손실이 모두 훨씬 작습니다. 빨간 직선이 점들의 가운데를 지납니다.
- **단위:** MAE는 "평균 몇 점 틀리는가"로 바로 읽습니다. MSE는 단위가 점수의 제곱이라 크기 비교에 씁니다.

### 3-2. 이상치 한 명이 손실을 얼마나 바꾸나

실제 데이터에는 평소와 동떨어진 값(이상치, outlier)이 섞이므로, 손실을 고르기 전에 이상치 하나가 두 손실을 얼마나 바꾸는지 알아야 합니다.

- 8시간 공부했지만 시험 날 아파서 20점을 받은 학생 한 명을 덧붙입니다.
- 같은 직선(`w=16`, `b=66`)으로 이상치 전(100명)과 이상치 후(101명)의 두 손실을 비교합니다.

In [ ]:
# 8시간 공부하고 20점을 받은 학생을 101번째로 덧붙입니다
x_raw_outlier = torch.cat([x_raw, torch.tensor([[8.0]], device=device)])
y_outlier = torch.cat([y, torch.tensor([[20.0]], device=device)])

# 원래 100명의 평균과 표준편차로 표준화해야 같은 직선(w=16, b=66)을 그대로 쓸 수 있습니다
x_outlier = (x_raw_outlier - x_raw.mean()) / x_raw.std()

# 같은 직선으로 이상치 전(100명)과 이상치 후(101명)를 예측합니다
y_pred = predict(x, 16.0, 66.0)
y_pred_outlier = predict(x_outlier, 16.0, 66.0)

outlier_pred = y_pred_outlier[-1].item()  # 마지막(101번째) 학생의 예측
print("덧붙인 학생의 점수 예측:", round(outlier_pred, 2))

In [ ]:
# 101명 각각의 벌점: 오차의 제곱(MSE)과 절댓값(MAE)
squared_errors = (y_pred_outlier - y_outlier) ** 2
absolute_errors = (y_pred_outlier - y_outlier).abs()

# 그 한 명의 벌점이 101명 벌점 합계에서 차지하는 비율
mse_share = (squared_errors[-1] / squared_errors.sum()).item()
mae_share = (absolute_errors[-1] / absolute_errors.sum()).item()

# 이상치 전후 손실
mse_before = loss_fn(y_pred, y).item()
mse_after = loss_fn(y_pred_outlier, y_outlier).item()
mae_before = mae_fn(y_pred, y).item()
mae_after = mae_fn(y_pred_outlier, y_outlier).item()

outlier_table = pd.DataFrame([
    {"손실": "MSE", "이상치 전": mse_before, "이상치 후": mse_after, "배수": mse_after / mse_before, "그 한 명의 비율(%)": mse_share * 100},
    {"손실": "MAE", "이상치 전": mae_before, "이상치 후": mae_after, "배수": mae_after / mae_before, "그 한 명의 비율(%)": mae_share * 100},
])

# 손실과 배수는 소수 둘째 자리, 비율은 소수 첫째 자리까지 봅니다
display(outlier_table.round({"이상치 전": 2, "이상치 후": 2, "배수": 2, "그 한 명의 비율(%)": 1}))

In [ ]:
# 왼쪽: 덧붙인 학생과 그 오차, 오른쪽: 그 한 명이 벌점 합계에서 차지하는 비율
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(x_raw.cpu(), y.cpu(), s=15, alpha=0.4, color="gray", label="학생 100명")
ax[0].scatter([8.0], [20.0], s=40, color="red", label="8시간 공부, 20점")
ax[0].plot(hours_grid.cpu(), predict(x_grid, 16.0, 66.0).cpu(), color="black", label="w=16, b=66")
ax[0].plot([8.0, 8.0], [20.0, outlier_pred], color="red", linestyle="--", label="그 학생의 오차")
ax[0].set_xlabel("공부시간(시간)")
ax[0].set_ylabel("점수")
ax[0].legend()

ax[1].bar(["MSE", "MAE"], [mse_share, mae_share], color=["red", "gray"])
ax[1].set_ylim(0, 1)
ax[1].set_ylabel("벌점 합계 중 그 한 명의 비율")
ax[1].set_title("101명 가운데 한 명이 차지하는 벌점")
plt.show()

**표와 그래프 읽기**
- **그 학생의 오차:** 직선은 8시간 공부한 학생을 82.79점으로 예측하므로 오차가 62.79점입니다(왼쪽 그래프의 빨간 점선). MSE 벌점은 62.79² = 3942.58, MAE 벌점은 62.79입니다.
- **MSE 행:** 23.58에서 62.38로 2.65배가 되고, 101명 벌점 합계 가운데 그 한 명이 62.6%를 차지합니다(오른쪽 빨간 막대).
- **MAE 행:** 4.08에서 4.66으로 1.14배가 되고, 그 한 명의 비율은 13.4%입니다(오른쪽 회색 막대).

**MSE로 학습하면 생기는 일**
- 학습은 손실을 줄이는 쪽으로 파라미터를 바꿉니다. MSE에서는 벌점의 62.6%가 그 한 명에게 몰려 있으므로, `w`와 `b`가 그 한 명의 오차를 줄이는 쪽으로 바뀌어 직선이 그 학생 쪽으로 내려옵니다.
- **이상치가 한 명일 때:** 직선이 그 학생 쪽으로 내려가면 나머지 100명의 오차가 커지므로, 직선은 조금만 내려옵니다.
- **같은 이상치가 여러 명일 때:** 그 학생들의 벌점이 합쳐져 직선이 크게 내려오고, 나머지 학생에게 가장 잘 맞는 직선에서 멀어집니다.

**실무에서 먼저 할 일: 이상치의 정체 확인**
- **입력 실수라면(점수를 잘못 옮겨 적음):** 데이터를 고칩니다.
- **실제로 일어난 일이라면(시험 날 결석·병결):** 그 사정을 입력 열로 넣거나, 예측하려는 상황과 다르면 학습 데이터에서 뺍니다.
- **그래도 드물게 큰 오차가 남는다면:** 큰 오차에 덜 민감한 MAE를 검토합니다.

**확인 퀴즈**: 점수 데이터에 입력 실수로 0점인 학생이 몇 명 섞였습니다. MSE로 학습한 직선과 MAE로 학습한 직선 가운데 어느 쪽의 예측이 더 많이 낮아질까요? 그리고 실무에서 먼저 할 일은 무엇일까요?

<details><summary>정답 보기</summary>

MSE로 학습한 직선입니다.

- **MSE:** 0점인 학생은 오차가 커서 제곱 벌점이 매우 큽니다. 이 벌점을 줄이려고 직선이 0점 쪽으로 내려갑니다.
- **MAE:** 벌점이 오차에 비례하므로 그 학생들이 손실에서 차지하는 비중이 MSE보다 작고, 직선도 덜 내려갑니다.
- **먼저 할 일:** 손실함수를 바꾸기 전에 0점이 입력 실수인지 확인해 데이터를 고칩니다.

</details>

## 4. 기울기와 autograd

손실을 계산할 수 있게 되었으니, 손실을 줄이려면 파라미터를 어느 쪽으로 바꿔야 하는지 구합니다.

### 4-1. 가중치 w에 따른 손실 곡선

파라미터를 어느 쪽으로 바꿀지 알려면, 먼저 파라미터 값에 따라 손실이 어떻게 달라지는지 봐야 합니다.

- 데이터가 정해져 있으면 손실은 파라미터만으로 정해지는 함수입니다.
- `b`를 점수 평균으로 고정하고 가중치 `w`만 바꿔 가며 손실을 그립니다. 표준화한 공부시간은 평균이 0이라 가장 좋은 `b`가 점수 평균과 같아서, `w` 하나만 보기에 알맞습니다.

In [ ]:
# b를 점수 평균으로 고정하고 w를 -10부터 40까지 바꿔 가며 손실(MSE)을 계산합니다
b_fixed = y.mean().item()
w_grid = torch.linspace(-10, 40, 101).tolist()
loss_curve = []
for w_value in w_grid:
    loss_curve.append(loss_fn(predict(x, w_value, b_fixed), y).item())

plt.figure(figsize=(6, 4))
plt.plot(w_grid, loss_curve)
plt.xlabel("가중치 w")
plt.ylabel("손실(MSE)")
plt.title("w에 따른 손실 (b는 점수 평균으로 고정)")
plt.show()

**그래프 읽기**
- 손실 곡선은 U자 모양이고 `w`가 약 16.6일 때 가장 작습니다(최솟값).
- 최솟값에서 왼쪽이든 오른쪽이든 멀어질수록 손실이 커집니다. 학습이 찾는 `w`는 이 최솟값 자리입니다.

### 4-2. autograd로 기울기 구하기

곡선 위의 한 점에서 손실이 어느 쪽으로 커지는지 알면, 반대로 가서 손실을 줄일 수 있습니다.

**기울기(gradient)**
- 파라미터를 아주 조금 바꿨을 때 손실이 바뀌는 비율입니다. 부호가 손실이 늘어나는 방향을 나타냅니다.
- **기울기가 음수:** 파라미터를 늘리면 손실이 줄어듭니다.
- **기울기가 양수:** 파라미터를 줄이면 손실이 줄어듭니다.
- 이 교안에서 '기울기'는 이 뜻(gradient)으로만 씁니다. 직선이 기운 정도는 '가중치 `w`'라고 부릅니다.

<img src="images/25_gradient_meaning.png" width="1050" alt="제목은 '기울기: w를 조금 바꿀 때 손실이 바뀌는 비율'입니다. 가로축 'w', 세로축 '손실'인 U자 곡선 위 'w = 5' 점에 작은 직각삼각형이 붙어 있습니다. 삼각형의 가로 변에는 'w를 0.01 늘림', 세로 변에는 '손실이 0.23 줄어듦'이 적혀 있습니다. 아래 상자에는 '기울기 = 손실 변화 ÷ w 변화 = -0.23 ÷ 0.01 = -23'과 '음수: w를 늘리면 손실이 줄어듭니다'가 적혀 있습니다.">

`b`를 점수 평균으로 고정한 손실 곡선에서 `w=5`의 기울기를 따라가 봅니다.

1. **w를 조금 늘리기:** `w`를 5에서 5.01로 0.01 늘리면 손실이 156.70에서 156.47로 0.23 줄어듭니다.
2. **비율 계산:** 손실 변화 ÷ `w` 변화 = -0.23 ÷ 0.01 = -23입니다. 이것이 `w=5`의 기울기입니다.
3. **뜻:** `w`를 아주 조금 늘리면 늘린 양의 약 23배만큼 손실이 줄어든다는 뜻입니다. 음수이므로 `w`를 늘려야 합니다.

이 값을 매번 손으로 계산하지 않고 PyTorch가 구하게 합니다.

**autograd: PyTorch가 기울기를 구하는 방법** ([PyTorch 자동 미분 튜토리얼](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html))
- **`requires_grad=True`:** 이 텐서로 하는 계산을 기록해 두라는 표시입니다. 기울기를 구할 파라미터에 붙입니다.
- **`loss.backward()`:** 기록된 계산을 손실에서 거꾸로 따라가며 기울기를 구합니다.
- **`.grad`:** 구한 기울기가 담기는 자리입니다. `w.grad`가 `w`의 기울기입니다.

최솟값(약 16.6)의 왼쪽(`w=5`), 근처(`w=16`), 오른쪽(`w=30`)에서 기울기를 구해 부호를 비교합니다.

In [ ]:
grad_rows = []
for w_start in [5.0, 16.0, 30.0]:
    # 기울기를 구할 파라미터는 requires_grad=True로 만듭니다
    w = torch.tensor([w_start], requires_grad=True, device=device)

    # 손실을 계산하고 backward()로 w.grad를 채웁니다
    loss = loss_fn(predict(x, w, b_fixed), y)
    loss.backward()
    grad_rows.append({"w": w_start, "손실": loss.item(), "w 기울기": w.grad.item()})

grad_table = pd.DataFrame(grad_rows)
display(grad_table.round(2))

In [ ]:
# 손실 곡선 위 세 점에 기울기만큼 기운 짧은 선(접선)을 그립니다
plt.figure(figsize=(6, 4))
plt.plot(w_grid, loss_curve, color="lightgray")
for row in grad_rows:
    # 점에서 왼쪽, 오른쪽으로 4만큼 갈 때 손실이 기울기 x 4만큼 바뀌는 선
    w_ends = [row["w"] - 4, row["w"] + 4]
    loss_ends = [row["손실"] - row["w 기울기"] * 4, row["손실"] + row["w 기울기"] * 4]
    plt.plot(w_ends, loss_ends, linewidth=2, label=f"w={row['w']}: 기울기 {round(row['w 기울기'], 1)}")
    plt.scatter([row["w"]], [row["손실"]], color="black", zorder=3)
plt.xlabel("가중치 w")
plt.ylabel("손실(MSE)")
plt.title("세 점의 기울기")
plt.legend()
plt.show()

**표와 그래프 읽기**
- **w=5:** 기울기 -23.0(음수). 선이 오른쪽 아래로 기울어 있어 `w`를 늘리면 손실이 줄어듭니다.
- **w=16:** 기울기 -1.22로 0에 가깝습니다. 선이 거의 평평해 `w`를 조금 바꿔도 손실이 거의 바뀌지 않습니다. 최솟값 근처입니다.
- **w=30:** 기울기 26.5(양수). 선이 오른쪽 위로 기울어 있어 `w`를 줄이면 손실이 줄어듭니다.

### 4-3. 기울기를 숫자로 확인하기

`.grad`가 정말 "조금 바꿨을 때 손실이 바뀌는 비율"인지 직접 확인해 봅니다.

- `w=5`에서 `w`를 0.01 늘린 손실과 0.01 줄인 손실을 계산합니다.
- 두 손실의 차이를 바꾼 폭(0.02)으로 나눈 값이 `w.grad`와 거의 같아야 합니다. 양쪽으로 바꾸면 한쪽만 바꿀 때보다 오차가 작습니다.

In [ ]:
# w=5에서 autograd로 구한 기울기
w = torch.tensor([5.0], requires_grad=True, device=device)
loss = loss_fn(predict(x, w, b_fixed), y)
loss.backward()

# w를 0.01 늘린 손실과 0.01 줄인 손실
nudge = 0.01
loss_up = loss_fn(predict(x, 5.0 + nudge, b_fixed), y).item()
loss_down = loss_fn(predict(x, 5.0 - nudge, b_fixed), y).item()
print("w=4.99일 때 손실:", round(loss_down, 4), "| w=5.01일 때 손실:", round(loss_up, 4))

# 손실 차이를 바꾼 폭(0.02)으로 나눈 값과 w.grad를 비교합니다
rate = (loss_up - loss_down) / (2 * nudge)
print("손실 변화 / 바꾼 폭:", round(rate, 2))
print("autograd 기울기 w.grad:", round(w.grad.item(), 2))

**결과 읽기**
- **손실 156.9257 -> 156.4657:** `w`를 4.99에서 5.01로 0.02 늘리자 손실이 약 0.46 줄었습니다.
- **손실 변화 / 바꾼 폭 -23.0, w.grad -23.0:** 두 값이 거의 같습니다. `backward()`가 구한 값이 "조금 바꿨을 때 손실이 바뀌는 비율"이라는 뜻입니다.

### 4-4. 역전파 직관: 체인룰

`backward()`가 계산을 거꾸로 따라가며 기울기를 구하는 규칙을 가장 작은 예로 확인합니다.

**체인룰(chain rule)**
- 계산이 여러 단계로 이어지면, 전체 기울기는 단계마다의 기울기를 곱한 값입니다.
- **역전파(backpropagation):** 손실에서 시작해 입력 쪽으로 체인룰을 차례로 적용하며 기울기를 구하는 방법입니다. `loss.backward()`가 하는 일이 역전파입니다.

<img src="images/11_chain_rule.png" width="1050" alt="제목은 '체인룰: 기울기를 뒤에서부터 곱해 갑니다'입니다. 위쪽 파란 화살표는 순전파로 x = 3에서 u = 2x + 1을 거쳐 u = 7, L = u²를 거쳐 L = 49로 갑니다. 아래쪽 보라 화살표는 역전파로 dL/du = 2u = 14, du/dx = 2를 뒤에서부터 전달합니다. 아래 수식 상자에는 dL/dx = dL/du × du/dx = 14 × 2 = 28이 적혀 있습니다.">

그림의 계산을 손으로 따라가 봅니다.

1. **순전파(값을 앞으로 계산):** `x = 3`이면 `u = 2 × 3 + 1 = 7`, `L = 7² = 49`입니다.
2. **뒤 단계의 기울기:** `L = u²`에서 `u`가 조금 늘 때 `L`은 `2u = 14`배 늡니다.
3. **앞 단계의 기울기:** `u = 2x + 1`에서 `x`가 조금 늘 때 `u`는 2배 늡니다.
4. **곱하기:** `x`의 기울기는 14 × 2 = 28입니다. autograd로 같은 값이 나오는지 확인합니다.

In [ ]:
# 그림과 같은 계산: x=3에서 u = 2x + 1, L = u의 제곱
x_value = torch.tensor(3.0, requires_grad=True)
u = 2 * x_value + 1
L = u ** 2

# L에서 거꾸로 기울기를 구합니다
L.backward()
print("u:", u.item(), "| L:", L.item())
print("x의 기울기(autograd):", x_value.grad.item())

autograd도 28을 구했습니다.

**우리 손실에 적용하면**
- MSE 손실도 여러 단계로 이어진 계산입니다: `w` → 예측(`w × x + b`) → 오차(예측 - 정답) → 오차의 제곱 → 평균.
- 체인룰로 뒤에서부터 곱하면 `w`의 기울기는 `평균(2 × 오차 × x)`가 됩니다. 제곱 단계가 `2 × 오차`를, 예측 단계가 `x`를 곱해 주기 때문입니다.
- 이 식으로 직접 계산한 값이 autograd로 구한 `w=5`의 기울기(-23.0)와 같은지 확인합니다.

In [ ]:
# 체인룰로 손으로 세운 식: w의 기울기 = 평균(2 x 오차 x 입력)
error = predict(x, 5.0, b_fixed) - y
print("체인룰 식으로 계산한 w 기울기:", round((2 * error * x).mean().item(), 2))

같은 -23.0입니다. 파라미터가 수백만 개인 신경망도 `backward()` 한 번이 이 곱셈을 모든 파라미터에 대해 해 줍니다. 그래서 이제부터는 기울기 식을 손으로 세우지 않고 `.grad`를 씁니다.

## 5. 경사하강

기울기가 손실이 늘어나는 방향을 알려 주므로, 그 반대 방향으로 파라미터를 바꾸는 일을 반복하면 손실이 줄어듭니다.

### 5-1. 갱신 식

**경사하강(gradient descent)**
- 기울기의 반대 방향으로 파라미터를 조금씩 바꾸는 갱신을 반복하는 방법입니다.
- **학습률(learning rate):** 기울기에 곱해, 한 번 갱신할 때 파라미터를 얼마나 바꿀지 정하는 수입니다.
- **갱신 크기:** 한 번 갱신에서 파라미터가 바뀌는 양으로, 학습률 × 기울기입니다.

```text
새 파라미터 = 지금 파라미터 - 학습률 × 기울기
```

- 말로 하면 "기울기에 학습률을 곱한 만큼, 기울기와 반대쪽으로 옮긴다"입니다. 기울기를 빼므로 기울기가 음수면 파라미터가 늘고, 양수면 줄어듭니다.

<img src="images/04_gradient_descent.png" width="1050" alt="제목은 '경사하강: 기울기 반대 방향으로 갱신'입니다. 가로축 w, 세로축 손실인 U자 곡선의 왼쪽 비탈 점에는 '기울기 음수 -> w를 늘린다', 오른쪽 비탈 점에는 '기울기 양수 -> w를 줄인다'가 붙어 있고, 두 점의 초록 화살표는 모두 가장 낮은 지점을 향합니다. 가장 낮은 지점의 별에는 '손실이 가장 작은 지점(최솟값)'이 붙어 있습니다. 아래 수식 상자에는 '새 w = w - 학습률 × 기울기'가 적혀 있습니다.">

학생 100명의 데이터로 첫 갱신이 일어나는 과정입니다. 학습률은 0.1입니다.

1. **시작:** `w=0`, `b=0`이면 모든 학생의 점수 예측이 0점이고 손실(MSE)은 4688.15입니다.
2. **기울기:** `w`의 기울기는 -32.9, `b`의 기울기는 -132.54입니다. 둘 다 음수이므로 둘 다 늘려야 합니다.
3. **갱신:** `w`는 0 - 0.1 × (-32.9) = 3.29, `b`는 0 - 0.1 × (-132.54) = 13.25가 됩니다.
4. **결과:** 손실이 3009.62로 줄어듭니다. 아래 코드가 이 과정을 그대로 실행합니다.

### 5-2. 손으로 한 번 갱신하기

옵티마이저를 쓰기 전에 갱신을 직접 써 보면 옵티마이저가 무엇을 대신하는지 알 수 있습니다.

- **`torch.zeros(1, requires_grad=True)`:** 0 하나가 든 파라미터 텐서를 만듭니다.
- **`with torch.no_grad():`:** 이 블록 안의 계산은 기록하지 않습니다. 갱신은 기울기를 구할 계산이 아니라 파라미터 값을 바꾸는 일이므로 이 안에서 합니다(밖에서 하면 PyTorch가 에러를 냅니다).

In [ ]:
# 파라미터 w, b를 0에서 시작합니다
w = torch.zeros(1, requires_grad=True, device=device)
b = torch.zeros(1, requires_grad=True, device=device)
lr = 0.1

# 예측과 손실을 계산하고, backward()로 기울기를 구합니다
loss = loss_fn(predict(x, w, b), y)
loss.backward()
print("갱신 전 손실:", round(loss.item(), 2), "| w 기울기:", round(w.grad.item(), 2), "| b 기울기:", round(b.grad.item(), 2))

# 기울기 반대 방향으로 갱신합니다
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
print("갱신 뒤 w:", round(w.item(), 2), "| b:", round(b.item(), 2), "| 손실:", round(loss_fn(predict(x, w, b), y).item(), 2))

**기울기를 비워야 하는 이유**
- PyTorch는 `backward()`를 부를 때마다 새 기울기를 `.grad`에 **더합니다**. 지난 갱신의 기울기가 남아 있으면 지금 위치의 기울기와 섞입니다.
- 그래서 다음 `backward()` 전에 `.grad.zero_()`로 0을 채워 비웁니다.

<img src="images/26_grad_accumulate.png" width="1050" alt="제목은 '.grad는 더해집니다: 다음 backward() 전에 비우기'입니다. 위 줄 '비우지 않으면'에는 '.grad' 상자에 'backward() 1번째: -32.9'와 'backward() 2번째: -26.38'이 쌓여 '합계 -59.28(섞인 값)'이 됩니다. 아래 줄 '비우면'에는 '.grad.zero_()'로 상자가 '0'이 된 뒤 'backward(): -26.38(지금 위치의 기울기)'만 남습니다.">

갱신을 한 번 한 뒤(`w=3.29`, `b=13.25`) 다시 기울기를 구하는 상황입니다.

1. **남아 있는 값:** 첫 `backward()`가 남긴 `w` 기울기 -32.9가 `.grad`에 그대로 있습니다.
2. **비우지 않고 backward():** 지금 위치의 기울기 -26.38이 더해져 -59.28이 됩니다. 이 값으로 갱신하면 지나치게 크게 움직입니다.
3. **비우고 backward():** 0에서 시작하므로 지금 위치의 기울기 -26.38만 남습니다. 아래 코드로 두 경우를 비교합니다.

In [ ]:
# 비우지 않고 한 번 더 backward()하면 새 기울기가 남아 있던 기울기에 더해집니다
loss = loss_fn(predict(x, w, b), y)
loss.backward()
print("비우지 않은 w 기울기:", round(w.grad.item(), 2))

# .grad.zero_()로 비운 뒤 다시 구하면 지금 w, b에서의 기울기만 남습니다
w.grad.zero_()
b.grad.zero_()
loss = loss_fn(predict(x, w, b), y)
loss.backward()
print("비운 뒤 w 기울기:", round(w.grad.item(), 2))

출력도 예시와 같이 비우지 않으면 -59.28(= -32.9 + (-26.38)), 비우면 -26.38입니다.

### 5-3. 갱신을 반복하기

한 번 갱신으로는 손실이 조금 줄 뿐이므로, 예측 → 손실 → 기울기 → 갱신 → 기울기 비우기를 100번 반복합니다.

- 갱신 0·1·3·10·100번째의 `w`, `b`, 손실을 기록해 둡니다.
- 기록한 `w`, `b`로 직선을 그려 학습이 직선을 어떻게 옮기는지 봅니다.

In [ ]:
# w, b를 0에서 다시 시작합니다
w = torch.zeros(1, requires_grad=True, device=device)
b = torch.zeros(1, requires_grad=True, device=device)
lr = 0.1
snapshot_rows = [{"갱신 횟수": 0, "w": w.item(), "b": b.item(), "손실": loss_fn(predict(x, w, b), y).item()}]

for step in range(1, 101):
    # 예측, 손실, 기울기
    loss = loss_fn(predict(x, w, b), y)
    loss.backward()

    # 갱신한 뒤 다음 반복을 위해 기울기를 비웁니다
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

    # 그림에 겹쳐 그릴 갱신 횟수에서 w, b, 손실을 기록합니다
    if step in [1, 3, 10, 100]:
        snapshot_rows.append({"갱신 횟수": step, "w": w.item(), "b": b.item(), "손실": loss_fn(predict(x, w, b), y).item()})

snapshot_table = pd.DataFrame(snapshot_rows)
display(snapshot_table.round(2))

In [ ]:
# 기록한 w, b로 그린 직선을 산점도에 겹쳐 봅니다
plt.figure(figsize=(7, 5))
plt.scatter(x_raw.cpu(), y.cpu(), s=15, alpha=0.4, color="gray")
for row in snapshot_rows:
    # 손실함수 절에서 만든 hours_grid, x_grid로 직선을 그리고, 범례에 그때의 손실을 붙입니다
    line = predict(x_grid, row["w"], row["b"])
    plt.plot(hours_grid.cpu(), line.cpu(), label=f"갱신 {row['갱신 횟수']}번 (손실 {round(row['손실'])})")
plt.xlabel("공부시간(시간)")
plt.ylabel("점수")
plt.ylim(-8, 105)  # 갱신 0번 직선(점수 0)이 그래프 아래 테두리와 겹쳐 가려지지 않게 아래 여백을 둡니다
plt.title("경사하강이 직선을 옮기는 모습")
plt.legend()
plt.show()

**표와 그래프 읽기**
- **갱신 0번:** `w=0`, `b=0`이라 모든 예측이 0점인 가로 직선이고 손실은 4688.15입니다.
- **갱신 1·3번:** 직선이 위로 올라오며 기울어지고, 손실이 3009.62 → 1247.12로 줄어듭니다.
- **갱신 10번:** `w` 14.79, `b` 59.15로 직선이 점들 사이로 들어왔고 손실은 77.07입니다.
- **갱신 100번:** `w` 16.62, `b` 66.27로 점들의 가운데를 지나고 손실은 23.13입니다. 손실함수 절에서 눈으로 고른 직선(`w=16`, `b=66`, 손실 23.58)보다 조금 더 작습니다.
- 처음에는 한 번에 크게 움직이고 최솟값에 가까워질수록 조금씩 움직입니다. 최솟값에 가까울수록 기울기가 작아져 갱신 크기도 작아지기 때문입니다.

### 5-4. 옵티마이저로 바꾸고 함수로 묶기

손으로 쓴 갱신과 기울기 비우기는 파라미터가 많아지면 파라미터 수만큼 줄이 늘어나므로, PyTorch의 옵티마이저(optimizer)에 맡깁니다.

**`torch.optim.SGD`** ([문서](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html))
- 경사하강 갱신 식을 그대로 실행하는 옵티마이저입니다. 만들 때 갱신할 파라미터 목록과 학습률을 넘깁니다: `torch.optim.SGD([w, b], lr=0.1)`.
- 손으로 쓴 코드와 옵티마이저 코드는 아래처럼 대응합니다.

| 하는 일 | 손으로 쓴 코드 | 옵티마이저 코드 |
|---|---|---|
| 기울기 비우기 | `w.grad.zero_()`, `b.grad.zero_()` | `optimizer.zero_grad()` |
| 기울기 구하기 | `loss.backward()` | `loss.backward()` (같음) |
| 갱신 | `with torch.no_grad():` 안에서 `w -= lr * w.grad` | `optimizer.step()` |

- 옵티마이저 코드는 보통 반복의 맨 앞에서 기울기를 비웁니다. 어느 쪽이든 다음 `backward()` 전에 비우면 됩니다.
- 학습률 절에서 학습률과 입력을 바꿔 가며 같은 학습을 여러 번 하므로, 반복을 함수 `train_line`으로 묶습니다.

In [ ]:
def train_line(x, y, lr, steps):
    """w, b를 0에서 시작해 SGD로 steps번 갱신하고 (w, b, loss_history)를 돌려줍니다."""
    w = torch.zeros(1, requires_grad=True, device=device)
    b = torch.zeros(1, requires_grad=True, device=device)
    optimizer = torch.optim.SGD([w, b], lr=lr)
    loss_history = []

    for step in range(steps):
        # 기울기 비우기 -> 예측과 손실 -> 기울기 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(predict(x, w, b), y)
        loss_history.append(loss.item())
        loss.backward()
        optimizer.step()

    # 마지막 갱신 뒤의 손실까지 기록해 loss_history[k]가 k번 갱신한 뒤의 손실이 되게 합니다
    with torch.no_grad():
        loss_history.append(loss_fn(predict(x, w, b), y).item())
    return w, b, loss_history


# 손으로 쓴 반복과 같은 조건(학습률 0.1, 100번)으로 학습해 결과를 비교합니다
w, b, loss_history = train_line(x, y, lr=0.1, steps=100)
print("손으로 쓴 반복 w:", round(snapshot_rows[-1]["w"], 4), "| b:", round(snapshot_rows[-1]["b"], 4))
print("SGD 옵티마이저 w:", round(w.item(), 4), "| b:", round(b.item(), 4))

In [ ]:
# 갱신마다 줄어드는 손실(학습 곡선)
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("갱신 횟수")
plt.ylabel("손실(MSE)")
plt.title("학습 곡선 (학습률 0.1)")
plt.show()

**결과 읽기**
- **w 16.6153, b 66.27:** 손으로 쓴 반복과 SGD 옵티마이저의 결과가 소수 넷째 자리까지 같습니다. 옵티마이저는 손으로 쓴 갱신을 대신 실행할 뿐입니다.
- **학습 곡선:** 처음 10번 안에 손실이 크게 줄고, 20번쯤부터 거의 평평합니다. 손실이 최솟값 근처에서 더 줄지 않는 이 상태를 수렴(convergence)이라고 합니다.

### 5-5. 학습된 w, b를 원래 단위로 읽기

`w`, `b`는 표준화한 공부시간 기준이라, 점수로 해석하려면 원래 단위(시간)로 바꿔야 합니다.

- 표준화한 공부시간 `x = (공부시간 - 평균) / 표준편차`를 직선에 넣고 정리하면 `점수 예측 = (w / 표준편차) × 공부시간 + (b - w × 평균 / 표준편차)`입니다.
- **`w / 표준편차`:** 공부 1시간당 늘어나는 점수입니다.
- **`b - w × 평균 / 표준편차`:** 공부시간이 0일 때의 점수 예측입니다.

In [ ]:
# 표준화 기준의 w, b를 원래 단위(시간)의 직선으로 바꿉니다
points_per_hour = w.item() / x_raw.std().item()
score_at_zero = b.item() - w.item() * x_raw.mean().item() / x_raw.std().item()
print("공부 1시간당 점수:", round(points_per_hour, 2))
print("공부시간 0일 때 점수 예측:", round(score_at_zero, 2))

**결과 읽기**
- **공부 1시간당 5.97점, 0시간일 때 35.91점:** 학습한 직선은 `점수 예측 = 5.97 × 공부시간 + 35.91`입니다.
- 이 데이터는 `점수 ≈ 35 + 6 × 공부시간 + 잡음`으로 만든 수업용 데이터라, 학습이 그 규칙을 거의 그대로 찾아냈습니다.

**확인 퀴즈**: 학습 루프에서 `optimizer.zero_grad()` 줄을 빠뜨렸습니다. 손실 기록은 어떻게 될까요?

<details><summary>정답 보기</summary>

기울기가 갱신마다 비워지지 않고 계속 더해집니다.

- 지난 갱신들의 기울기가 쌓인 값으로 갱신하므로 갱신 크기가 지금 위치의 기울기보다 훨씬 커집니다.
- 그래서 파라미터가 최솟값을 지나쳐 반대쪽으로 가고, 손실이 줄었다 늘었다를 반복하며 최솟값에 머물지 못합니다.
- 손실 기록이 이상하게 오르내리면 `zero_grad()`가 루프 안에 있는지부터 확인합니다.

</details>

## 6. 학습률

같은 데이터와 같은 모델이라도 학습률에 따라 손실이 줄어드는 속도와 수렴 여부가 달라지므로, 학습률을 고르는 기준이 필요합니다.

### 6-1. 느림·수렴·발산

학습률을 고르려면 학습률이 너무 작을 때와 너무 클 때 손실 기록이 어떻게 달라지는지 먼저 알아야 합니다.

- **느림:** 학습률이 너무 작으면 갱신 크기가 작아, 손실이 최솟값 근처까지 줄어드는 데 갱신이 많이 필요합니다.
- **수렴:** 손실이 최솟값 근처에서 더 줄지 않게 됩니다.
- **발산(divergence):** 학습률이 너무 크면 파라미터가 최솟값을 지나쳐 반대쪽에서 손실이 더 커지고, 이 일이 갱신마다 반복되며 손실이 끝없이 커집니다.

<img src="images/05_learning_rate.png" width="1050" alt="제목은 '학습률: 기울기에 곱해 갱신 크기를 정하는 수'이고 같은 U자 손실 곡선 판 세 개가 나란히 있습니다. '너무 작다' 판에서는 점들이 촘촘히 내려가지만 최솟값에 아직 닿지 못해 '최솟값까지 오래 걸림'입니다. '수렴' 판에서는 큰 간격으로 세 번 갱신해 최솟값 근처에 닿아 '몇 번 갱신하면 최솟값 근처'입니다. '너무 크다' 판에서는 점이 갱신마다 반대쪽 비탈로 건너가며 점점 높아져 '최솟값을 지나쳐 손실이 커짐(발산)'입니다. 아래 결론 상자에는 '손실이 커지거나 nan이 나오면 학습률부터 줄입니다'가 적혀 있습니다.">

학습률 세 개로 30번씩 학습해 손실 기록을 비교합니다. 이 데이터의 최솟값은 경사하강으로 수렴한 손실 23.13입니다.

1. **lr=0.01:** 갱신 크기가 작아 30번 뒤에도 손실이 최솟값과 멉니다(느림).
2. **lr=0.1:** 30번 안에 최솟값 근처까지 줄어듭니다(수렴).
3. **lr=1.05:** 반대쪽에서 손실이 더 커지는 일이 갱신마다 반복됩니다(발산).

**갱신 크기 = 학습률 × 기울기**
- 같은 기울기라도 학습률에 따라 한 번에 움직이는 거리가 달라집니다. 첫 갱신의 `b`로 보면 차이가 바로 보입니다.

<img src="images/27_step_size.png" width="1050" alt="제목은 '갱신 크기 = 학습률 × 기울기'입니다. 가로 수직선 'b'에 0, 66.27, 139.17 눈금이 있고 66.27에는 '가장 좋은 b(점수 평균)' 별이 있습니다. 0에서 출발하는 화살표 셋이 '학습률 0.01: 1.33(느림)', '학습률 0.1: 13.25(수렴)', '학습률 1.05: 139.17(지나침)'까지 갑니다. 위 띠에는 'b의 기울기 -132.54에서 첫 갱신'이 적혀 있습니다.">

`b=0`에서 `b`의 기울기는 -132.54입니다. 첫 갱신에서 `b`가 움직이는 거리입니다.

1. **lr=0.01:** 0.01 × 132.54 = 1.33만큼 움직여, 가장 좋은 `b`(66.27)까지 한참 남습니다.
2. **lr=0.1:** 13.25만큼 움직여 66.27 쪽으로 다가갑니다.
3. **lr=1.05:** 139.17만큼 움직여 66.27을 지나쳐 반대쪽으로 더 멀리 가 버립니다.

In [ ]:
# 느림, 수렴, 발산 세 경우를 보려고 학습률 0.01, 0.1, 1.05로 30번씩 학습합니다
_, _, slow_history = train_line(x, y, lr=0.01, steps=30)
_, _, good_history = train_line(x, y, lr=0.1, steps=30)
_, _, diverge_history = train_line(x, y, lr=1.05, steps=30)
lr_history_table = pd.DataFrame({"lr=0.01": slow_history, "lr=0.1": good_history, "lr=1.05": diverge_history})

# 31행 전체는 아래 그래프로 보고, 표에는 변화가 보이는 갱신 횟수만 고릅니다
display(lr_history_table.loc[[0, 1, 2, 3, 5, 10, 20, 30]].round(1))

In [ ]:
# 발산한 손실은 너무 커서 로그 축이어야 세 곡선이 함께 보입니다
plt.figure(figsize=(6, 4))
plt.plot(slow_history, label="lr=0.01 (느림)")
plt.plot(good_history, label="lr=0.1 (수렴)")
plt.plot(diverge_history, label="lr=1.05 (발산)")
plt.yscale("log")
plt.xlabel("갱신 횟수")
plt.ylabel("손실(MSE, 로그 축)")
plt.title("학습률에 따른 손실 변화")
plt.legend()
plt.show()

**표와 그래프 읽기**
- **lr=0.01 (느림):** 손실이 갱신마다 줄지만 30번 행에서도 1412.2입니다.
- **lr=0.1 (수렴):** 10번 행 77.1, 20번 행 23.8, 30번 행 23.1로 최솟값(23.13) 근처에서 더 줄지 않습니다.
- **lr=1.05 (발산):** 4688.1 → 5655.3 → 6823.5로 갱신마다 커져 30번 행은 1363380.4(약 136만)입니다. 로그 축에서 거의 직선으로 올라가는 것은 갱신마다 손실이 일정한 비율(약 1.2배)로 커진다는 뜻입니다.
- **실무 신호:** 학습 로그의 손실이 갱신마다 커지거나 `nan`(not a number, 너무 커져 계산할 수 없게 된 값)이 찍히면 학습률부터 줄입니다.

### 6-2. 입력의 크기와 학습률

학습률 하나를 `w`와 `b`에 똑같이 쓰므로, 입력이 크면 `b`에 알맞은 학습률이 `w`에는 너무 클 수 있습니다.

- **w와 b가 예측을 바꾸는 크기:** `w`를 1 바꾸면 예측은 입력값만큼 바뀌고, `b`를 1 바꾸면 예측은 1만큼 바뀝니다. 원래 공부시간은 평균 약 5시간이라 `w`를 조금만 바꿔도 예측이 크게 바뀝니다.
- **기울기에도 입력이 곱해짐:** 체인룰로 구한 대로 `w`의 기울기는 평균(2 × 오차 × 입력)이고, `b`의 기울기에는 입력이 곱해지지 않습니다. 그래서 입력이 크면 `w`의 갱신 크기도 커집니다.

<img src="images/28_input_scale.png" width="1050" alt="제목은 '입력이 크면 w를 조금만 바꿔도 예측이 크게 바뀝니다'입니다. 같은 세로 눈금의 판 두 개가 있습니다. 왼쪽 '원래 공부시간' 판에는 'w를 1 바꿀 때 예측 변화' 보라 막대 5.08이 'b를 1 바꿀 때 예측 변화' 회색 막대 1보다 약 다섯 배 높습니다. 오른쪽 '표준화한 공부시간' 판에는 보라 막대 0.84와 회색 막대 1이 거의 같은 높이입니다. 아래 상자에는 '표준화하면 학습률 하나로 w와 b를 함께 학습합니다'가 적혀 있습니다.">

- 표준화하지 않은 공부시간(`x_raw`, 0.5~10시간)을 그대로 넣으면 어떻게 되는지 비교합니다.
- 먼저 학습을 시작하는 `w=0`, `b=0`에서 두 입력의 기울기를 봅니다.

In [ ]:
# 학습을 시작하는 w=0, b=0에서 원래 공부시간과 표준화한 공부시간의 기울기를 비교합니다
input_grad_rows = []
for input_name, x_input in [("원래 공부시간", x_raw), ("표준화한 공부시간", x)]:
    w = torch.zeros(1, requires_grad=True, device=device)
    b = torch.zeros(1, requires_grad=True, device=device)
    loss = loss_fn(predict(x_input, w, b), y)
    loss.backward()

    # 입력 절댓값 평균: w를 1 바꿀 때 예측이 평균적으로 바뀌는 크기
    input_grad_rows.append({"입력": input_name, "입력 절댓값 평균": x_input.abs().mean().item(), "w 기울기": w.grad.item(), "b 기울기": b.grad.item()})

input_grad_table = pd.DataFrame(input_grad_rows)
display(input_grad_table.round(2))

**표 읽기**
- **입력 절댓값 평균:** 원래 공부시간은 5.08, 표준화한 공부시간은 0.84입니다. `w`를 1 바꿀 때 예측이 평균적으로 바뀌는 크기입니다. `b`를 1 바꾸면 예측은 늘 1만큼 바뀝니다.
- **원래 공부시간 행:** `w`를 1 바꾸면 예측이 평균 5.08만큼 바뀌어 `b`보다 약 5배 크게 바뀝니다. 기울기에도 입력이 곱해져 `w` 기울기(-765.05)가 `b` 기울기(-132.54)의 약 5.8배입니다. 그래서 `b`에 알맞은 학습률이 `w`에는 너무 큽니다.
- **표준화한 공부시간 행:** `w` 기울기 -32.9, `b` 기울기 -132.54로 기울기 크기는 약 4배 차이가 납니다. 그래도 입력이 1 안팎이라 `w`를 1 바꿀 때 예측이 바뀌는 크기(0.84)가 `b`(1)와 비슷해, 학습률 하나로 둘을 함께 학습할 수 있습니다. `b`의 기울기는 입력과 상관없이 같습니다.
- 원래 공부시간에서 학습률을 `b`에 맞추면 `w`의 갱신 크기가 너무 크고, `w`에 맞추면 `b`의 갱신 크기가 너무 작습니다. 두 학습률로 확인합니다.

In [ ]:
# 같은 1000번 갱신을 세 가지로 학습합니다: 원래 공부시간(학습률 0.1, 0.01), 표준화한 공부시간(학습률 0.1)
_, _, raw_fast_history = train_line(x_raw, y, lr=0.1, steps=1000)
_, _, raw_slow_history = train_line(x_raw, y, lr=0.01, steps=1000)
_, _, standard_history = train_line(x, y, lr=0.1, steps=1000)

# 100번 뒤와 1000번 뒤의 손실을 표로 비교합니다
input_table = pd.DataFrame([
    {"입력": "원래 공부시간", "학습률": 0.1, "100번 뒤 손실": raw_fast_history[100], "1000번 뒤 손실": raw_fast_history[1000]},
    {"입력": "원래 공부시간", "학습률": 0.01, "100번 뒤 손실": raw_slow_history[100], "1000번 뒤 손실": raw_slow_history[1000]},
    {"입력": "표준화한 공부시간", "학습률": 0.1, "100번 뒤 손실": standard_history[100], "1000번 뒤 손실": standard_history[1000]},
])
display(input_table.round(2))

In [ ]:
# 학습률 0.1은 손실이 nan이 된 뒤로는 그려지지 않으므로, 수렴한 두 경우의 손실 기록을 비교합니다
plt.figure(figsize=(6, 4))
plt.plot(raw_slow_history, label="원래 공부시간, 학습률 0.01")
plt.plot(standard_history, label="표준화한 공부시간, 학습률 0.1")
plt.yscale("log")
plt.xlabel("갱신 횟수")
plt.ylabel("손실(MSE, 로그 축)")
plt.title("입력을 표준화하면 빨리 수렴합니다")
plt.legend()
plt.show()

**표와 그래프 읽기**
- **원래 공부시간, 학습률 0.1:** 손실이 `NaN`입니다. 기울기가 큰 `w`의 갱신 크기가 너무 커서 발산하다가 계산할 수 없을 만큼 커졌습니다.
- **원래 공부시간, 학습률 0.01:** 발산하지는 않지만 100번 뒤 손실이 132.29이고, 1000번 뒤에야 23.16으로 최솟값 근처에 옵니다. 그래프의 파란 선은 처음 몇 번에 300 근처까지 빠르게 내려간 뒤(기울기가 큰 `w`가 먼저 맞춰짐) 아주 천천히 내려갑니다. `w`에 맞춘 작은 학습률이라 `b`의 갱신 크기가 너무 작기 때문입니다.
- **표준화한 공부시간, 학습률 0.1:** 100번 뒤 이미 23.13입니다. 그래프의 주황 선은 처음 20번 안에 최솟값 근처까지 내려갑니다.
- 입력을 표준화하면 입력이 1 안팎이 되어, `w`를 1 바꿀 때 예측이 바뀌는 크기가 `b`를 1 바꿀 때와 비슷해집니다. 그래서 학습률 하나로 둘 다 빨리 학습됩니다. 신경망 학습 전에 입력을 표준화하는 이유입니다.

### 6-3. 학습률 고르기

수렴하는 학습률의 범위는 데이터·모델·입력 크기마다 다르므로, 후보를 넓게 시도하고 그 가운데 하나를 고르는 순서를 정해 둡니다.

**고르는 순서**
1. **10배 간격으로 넓게 시도:** 0.001, 0.01, 0.1, 1.0처럼 10배 간격 후보를 같은 횟수(여기서는 30번) 갱신한 뒤의 손실로 비교합니다. 학습률은 신경망 학습에서 가장 중요한 하이퍼파라미터(사람이 학습 전에 정하는 설정)인 경우가 많고, 후보를 로그 간격(일정한 배수)으로 늘어놓고 찾는 방법이 권장됩니다([Bengio, 2012](https://arxiv.org/abs/1206.5533), 3.1.1절·3.3절).
2. **발산 경계 찾기:** 수렴한 후보와 수렴하지 않은 후보 사이 값을 더 시도해, 손실이 줄지 않기 시작하는 학습률(발산 경계)을 찾습니다.
3. **경계에서 먼 값 고르기:** 정한 횟수 안에 수렴한 후보 가운데 발산 경계에서 가장 먼 값을 고릅니다. 데이터가 바뀌어 경계가 내려와도 발산하지 않게 여유를 두기 위해서입니다.

- **가장 좋은 학습률과 경계:** Bengio(2012, 3.1.1절)는 가장 좋은 학습률이 발산하지 않는 가장 큰 학습률의 2배 이내에 있는 경우가 많다고 적습니다. 이 교안은 데이터가 바뀌어도 발산하지 않도록 여유를 더 두는 쪽을 고릅니다.

In [ ]:
# 10배 간격 후보 0.001, 0.01, 0.1, 1.0에 0.1과 1.0 사이의 0.3, 0.9를 더해 작은 값부터 시도합니다
lr_rows = []
for lr in [0.001, 0.01, 0.1, 0.3, 0.9, 1.0]:
    _, _, history = train_line(x, y, lr, steps=30)
    lr_rows.append({"학습률": lr, "30번 갱신 뒤 손실": history[-1]})

lr_table = pd.DataFrame(lr_rows)
display(lr_table.round({"30번 갱신 뒤 손실": 2}))

In [ ]:
# 학습률마다 30번 갱신 뒤 손실을 점으로 찍습니다(두 축 모두 로그 축)
plt.figure(figsize=(6, 4))
plt.plot(lr_table["학습률"], lr_table["30번 갱신 뒤 손실"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xticks([0.001, 0.01, 0.1, 1.0], ["0.001", "0.01", "0.1", "1.0"])  # 눈금을 10의 거듭제곱 대신 소수로 표시합니다
plt.xlabel("학습률(로그 축)")
plt.ylabel("30번 갱신 뒤 손실(로그 축)")
plt.title("학습률 후보별 30번 갱신 뒤 손실")
plt.show()

<img src="images/22_lr_search.png" width="1050" alt="제목은 '학습률 고르기: 10배 간격으로 넓게, 발산 경계에서 멀리'입니다. 0.001, 0.01, 0.1, 1 눈금자 위에서 0.001부터 0.1 앞까지는 노랑 '느림', 0.1부터 0.9까지는 초록 '30번 안에 수렴', 1 이상은 빨강 '수렴하지 않음(발산 경계)' 구간입니다. 0.1 눈금에 '선택' 깃발이 꽂혀 있습니다. 아래 단계 카드는 1 '10배 간격으로 후보 시도', 2 '발산이 시작되는 값 찾기', 3 '수렴한 후보 가운데 발산 경계에서 가장 먼 값 고르기' 순서입니다.">

표와 그래프로 순서를 따라가 봅니다(최솟값은 23.13). 그래프에서 손실이 최솟값 근처(23.13)에 머문 구간(0.1~0.9)이 수렴한 후보이고, 양쪽 끝으로 갈수록 손실이 다시 큽니다.

1. **10배 간격 후보:** 0.001(4160.43)과 0.01(1412.22)은 느림이고, 0.1(23.14)만 최솟값 근처까지 줄었습니다. 1.0(4496.14)은 30번 갱신해도 손실이 거의 줄지 않았습니다.
2. **발산 경계 찾기:** 0.1과 1.0 사이의 0.3(23.13)과 0.9(23.14)는 수렴했고, 1.0은 수렴하지 않았으며 앞에서 본 1.05는 발산했습니다. 발산 경계는 1.0 근처입니다. 1.0에서는 첫 갱신에서 `b`가 0 - 1.0 × (-132.54) = 132.54가 되어 최솟값(66.27)을 같은 거리만큼 넘어가고, 이후로도 0과 132.54 사이를 오가므로 손실이 거의 줄지 않습니다.
3. **경계에서 먼 값 고르기:** 30번 안에 수렴한 0.1, 0.3, 0.9 가운데 발산 경계(1.0)에서 가장 먼 **0.1**을 고릅니다. 0.3은 30번 뒤 손실이 조금 더 낮지만(23.13) 경계(1.0)와 약 3배 차이뿐이라 고르지 않습니다. 0.9는 경계에 너무 가까워 데이터가 조금만 바뀌어도 발산할 수 있습니다.

**확인 퀴즈**: 한 실험은 학습 로그의 손실이 처음 몇 번은 줄다가 갑자기 커지더니 `nan`이 찍혔습니다. 다른 실험은 손실이 줄긴 하지만 1,000번 갱신해도 아주 조금씩만 줄어듭니다. 두 실험에서 각각 무엇을 바꿔야 할까요?

<details><summary>정답 보기</summary>

- **첫 실험(발산):** 학습률이 너무 커서 파라미터가 최솟값을 지나쳐 손실이 커진 경우입니다. 학습률을 줄입니다(예: 10분의 1). 입력을 표준화하지 않았다면 표준화도 확인합니다.
- **둘째 실험(느림):** 학습률이 너무 작아 갱신 크기가 작은 경우입니다. 발산하지 않는 범위에서 학습률을 키웁니다(예: 10배). 입력마다 크기가 크게 다르면 표준화하면 같은 학습률로 더 빨리 수렴합니다.

</details>

## 7. 직선 모델을 nn.Linear로 바꾸기

지금까지 `w`, `b`를 직접 만들었지만, 신경망은 이런 직선을 수천 개 씁니다. 그래서 PyTorch가 제공하는 층 `nn.Linear`로 같은 직선을 만들어 봅니다.

**`nn.Linear(입력 수, 출력 수)`** ([문서](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html))
- 입력에 가중치를 곱하고 절편을 더하는 층입니다. `nn.Linear(1, 1)`은 입력 1개, 출력 1개라 우리가 만든 직선 `w × x + b`와 같습니다.
- **`.weight`, `.bias`:** 층 안의 파라미터입니다. 우리가 만든 `w`, `b`에 해당하고, 처음부터 `requires_grad=True`입니다. 시작 값은 무작위로 정해집니다.
- **`model.parameters()`:** 층의 파라미터 목록을 돌려줍니다. 옵티마이저에 `[w, b]` 대신 넘깁니다.

<img src="images/23_linear_neuron.png" width="1050" alt="제목은 '직선 하나 = nn.Linear(1, 1) = 뉴런 하나'입니다. 왼쪽 '우리가 만든 직선' 판에는 공부시간 x가 '× w'와 '+ b' 상자를 지나 점수 예측으로 가는 화살표가 있습니다. 오른쪽 'nn.Linear(1, 1)' 판에는 입력 1개가 원 하나(뉴런)로 들어가 출력 1개로 나오고, 원 옆에 'weight = w', 'bias = b'가 붙어 있습니다. 두 판 사이에 '='가 있습니다. 아래 띠에는 '교안 02: 뉴런을 여러 개 쌓아 손글씨 숫자를 분류합니다'가 적혀 있습니다.">

In [ ]:
# 시작 값을 실행마다 같게 고정합니다(nn.Linear의 시작 값은 무작위)
torch.manual_seed(0)
model = nn.Linear(1, 1).to(device)
print("weight:", model.weight)
print("bias:", model.bias)

# model(x)와 predict(x, weight, bias)가 같은 계산인지 앞 세 명으로 확인합니다
print("model(x):", [round(v, 4) for v in model(x)[:3].flatten().tolist()])
print("predict(x, weight, bias):", [round(v, 4) for v in predict(x, model.weight, model.bias)[:3].flatten().tolist()])

**결과 읽기**
- **`Parameter containing:`:** 학습할 파라미터로 표시된 텐서라는 뜻입니다.
- **weight는 대괄호 두 겹, bias는 한 겹:** `nn.Linear`는 입력이 여러 개여도 쓸 수 있게 가중치를 (출력 수, 입력 수) 모양으로 둡니다. 입력 1개, 출력 1개라 (1, 1)입니다. `requires_grad=True`가 이미 붙어 있습니다.
- **두 예측이 같음:** `model(x)`는 `x × weight + bias`를 계산합니다. 우리가 만든 `predict`와 같은 계산입니다.

이제 `train_line`과 같은 학습을 `nn.Linear`로 합니다. 바뀌는 곳은 예측(`model(x)`)과 옵티마이저에 넘기는 파라미터(`model.parameters()`) 두 군데입니다.

In [ ]:
# 옵티마이저에 [w, b] 대신 model.parameters()를 넘깁니다
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# train_line과 같은 반복: 기울기 비우기 -> 예측과 손실 -> 기울기 -> 갱신
for step in range(100):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()

print("nn.Linear weight:", round(model.weight.item(), 4), "| bias:", round(model.bias.item(), 4), "| 손실:", round(loss_fn(model(x), y).item(), 2))

**결과 읽기**
- **weight 16.6153, bias 66.27, 손실 23.13:** 시작 값(-0.0075, 0.5364)은 0이 아니었지만 `train_line`의 `w`, `b`와 같은 값으로 수렴했습니다. 이 손실 곡선에는 최솟값이 하나뿐이라 어디서 시작해도 같은 자리로 갑니다.

**다음 교안으로**
- `nn.Linear(1, 1)`은 입력에 가중치를 곱하고 절편을 더해 출력 하나를 내는 **뉴런 하나**입니다.
- 교안 02에서는 이 뉴런을 여러 개 쌓은 신경망(DNN)으로 손글씨 숫자를 분류합니다. 학습 코드는 오늘과 같은 `zero_grad()` → 손실 → `backward()` → `step()`입니다.

## 8. 🖐️ 함께 따라하기: 자동차 무게로 연비 예측하기

본실습에서 만든 `train_line`을 다른 분야의 데이터에 적용해, 입력 표준화가 학습을 어떻게 바꾸는지와 학습된 `w`를 실무 숫자로 읽는 법을 연습합니다. 본실습은 학생 성적, 따라하기는 자동차 데이터입니다.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **부분 작성:** 셀에 남은 `...`만 주석에 맞춰 채웁니다. 나머지 코드는 제공합니다.
- `...`가 남은 채로 실행하면 에러가 나거나 빈 결과가 나옵니다. 작성 셀을 완성해 실행하면 같은 셀에서 결과를 볼 수 있습니다.
- 먼저 직접 작성해 봅니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

### 8-1. 질문과 작성 목표

작성 셀을 채우기 전에 무엇을 만들고 어떻게 확인하는지 정리합니다.

- **배경:** 자동차 398대의 무게(kg)와 연비(km/L)가 있습니다([UCI Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg), CC BY 4.0, 무게는 kg, 연비는 km/L로 바꿨습니다). 무거운 차일수록 연비가 낮다고 보고 직선 `연비 예측 = w × 무게 + b`를 학습합니다.
- **작성:** 무게를 원래 단위(kg) 그대로 본실습에서 고른 학습률 0.1로 학습해 보고, 무게를 표준화해 같은 학습률로 다시 학습합니다.
- **확인:** 학습된 `w`를 "무게가 100kg 늘 때 연비가 몇 km/L 바뀌는가"로 바꿔 읽습니다.
- **힌트:** 텐서 만들기와 표준화는 2절, 학습은 `train_line`, 원래 단위로 읽기는 5-5절과 같은 방법입니다.

In [ ]:
# 자동차 398대: 차종, 연식, 무게(kg), 연비(km/L)
cars = pd.read_csv(data_dir / "따라하기" / "auto_mpg.csv")
print("행 수:", len(cars))
display(cars.head())

In [ ]:
# 무게가 무거울수록 연비가 낮아지는지 산점도로 봅니다
plt.figure(figsize=(6, 4))
plt.scatter(cars["무게"], cars["연비"], s=10, alpha=0.5)
plt.xlabel("무게(kg)")
plt.ylabel("연비(km/L)")
plt.title("자동차 무게와 연비")
plt.show()

산점도의 점들이 오른쪽 아래로 늘어섭니다. 무게가 무거울수록 연비가 낮으므로 학습된 `w`는 음수가 되어야 합니다.

### 8-2. 원래 단위로 학습하기

입력을 표준화하지 않으면 이 데이터에서 무슨 일이 일어나는지 먼저 확인합니다. 무게는 732~2331kg으로 공부시간(0.5~10시간)보다 훨씬 큰 수입니다.

In [ ]:
# 🖐️ 함께 따라하기: 원래 단위(kg)의 무게로 직선을 학습하고 손실 기록을 확인하세요

# 1) 무게와 연비를 본실습과 같은 모양, 같은 자료형의 텐서로 만들어 계산 장치에 올리기
mass_raw = ...
efficiency = ...

# 2) 본실습에서 고른 학습률 0.1로 100번 학습하기
w_raw, b_raw, raw_history = ...

print("처음 손실 네 개:", raw_history[:4])
print("마지막 손실:", raw_history[-1])

**결과 읽기** (작성 셀을 실행한 뒤 확인합니다)
- **처음 손실 네 개:** 갱신 전 110.95에서 첫 갱신 뒤 1.2e+13(12조) 수준으로 커지고, 갱신마다 더 커지면 발산한 것입니다.
- **마지막 손실 `nan`:** 손실이 계산할 수 없을 만큼 커졌습니다. 본실습에서 고른 학습률 0.1도 입력의 크기가 다르면 발산합니다.

### 8-3. 표준화해서 학습하기

학습률은 그대로 두고 입력만 표준화해 다시 학습합니다.

In [ ]:
# 🖐️ 함께 따라하기: 무게를 표준화한 뒤 같은 학습률, 같은 횟수로 다시 학습하세요

# 1) 무게를 평균 0, 표준편차 1로 표준화하기
mass = ...

# 2) 표준화한 무게로 학습하기
w_fit, b_fit, fit_history = ...

print("마지막 손실:", round(fit_history[-1], 2))
print("학습된 w:", round(w_fit.item(), 4), "| b:", round(b_fit.item(), 4))

In [ ]:
# 학습된 직선을 산점도에 겹쳐 봅니다(작성 셀을 실행한 뒤 실행합니다)
# 직선을 그릴 무게 700~2400kg 50개를 (50, 1) 모양으로 만들고, 자동차 데이터와 같은 평균, 표준편차로 표준화합니다
mass_grid_raw = torch.linspace(700, 2400, 50, device=device).reshape(50, 1)
mass_grid = (mass_grid_raw - mass_raw.mean()) / mass_raw.std()

plt.figure(figsize=(6, 4))
plt.scatter(mass_raw.cpu(), efficiency.cpu(), s=10, alpha=0.4)
plt.plot(mass_grid_raw.cpu(), predict(mass_grid, w_fit.item(), b_fit.item()).cpu(), color="red")
plt.xlabel("무게(kg)")
plt.ylabel("연비(km/L)")
plt.title("표준화한 무게로 학습한 직선")
plt.show()

In [ ]:
# 무거운 차(2,200kg 이상)만 골라 실제 연비 평균과 직선의 예측 평균을 비교합니다
heavy_cars = cars[cars["무게"] >= 2200]
heavy_mass_raw = torch.tensor(heavy_cars[["무게"]].to_numpy(), dtype=torch.float32, device=device)
heavy_mass = (heavy_mass_raw - mass_raw.mean()) / mass_raw.std()
heavy_pred = predict(heavy_mass, w_fit.item(), b_fit.item())

print("무거운 차 수:", len(heavy_cars))
print("실제 연비 평균(km/L):", round(heavy_cars["연비"].mean(), 2))
print("직선의 예측 평균(km/L):", round(heavy_pred.mean().item(), 2))

**결과 읽기** (작성 셀을 실행한 뒤 확인합니다)
- **마지막 손실 3.4 근처:** 발산하지 않고 수렴했습니다. 학습률은 같고 입력만 표준화했습니다.
- **w가 음수(-2.76 근처):** 무게가 표준편차 하나만큼 무거워지면 연비가 약 2.76km/L 낮아진다는 뜻입니다. `b`(10.0 근처)는 평균 무게인 차의 연비 예측이라 연비 평균과 거의 같습니다.
- **그래프:** 빨간 직선이 오른쪽 아래로 내려가며 점들의 가운데를 지납니다. 다만 점들은 무거워질수록 완만해지는 휘어진 모양이라, 양 끝에서 직선과 어긋납니다.
- **무거운 차 6대:** 실제 연비 평균은 5.1km/L인데 직선은 3.43km/L로 낮게 예측합니다. 직선 하나로는 휘어진 관계를 다 맞추지 못합니다. 교안 02의 은닉층(ReLU)이 이런 휘어진 관계를 맞춥니다.

### 8-4. 학습된 w를 원래 단위로 읽기

"표준편차 하나만큼"은 실무에서 바로 쓰기 어려우므로, 정비사나 구매 담당자가 바로 이해하는 "100kg당 연비 변화"로 바꿉니다.

In [ ]:
# 🖐️ 함께 따라하기: 학습된 w를 무게 100kg당 연비 변화(km/L)로 바꾸세요

# 1) 표준화 기준의 w를 무게 1kg당 연비 변화로 바꾸기
per_kg = ...

# 2) 100kg당 연비 변화로 바꾸기
per_100kg = ...

print("무게 100kg이 늘 때 연비 변화(km/L):", round(per_100kg, 2))

**결과 읽기** (작성 셀을 실행한 뒤 확인합니다)
- **-0.72 근처:** 무게가 100kg 늘 때마다 연비가 약 0.72km/L 떨어진다고 예측합니다.

**생각해 보기**: 무게가 1,000kg인 차와 1,500kg인 차의 연비는 몇 km/L 차이 난다고 예측할까요? 그리고 본실습에서 수렴한 학습률 0.1이 이 데이터의 원래 단위에서는 왜 발산했을까요?

<details><summary>정답 보기</summary>

- **연비 차이:** 500kg 차이는 100kg의 5배이므로 0.72 × 5 = 약 3.6km/L 차이 납니다. 1,500kg 차가 약 3.6km/L 낮다고 예측합니다.
- **발산한 이유:** 무게(732~2331)가 공부시간(0.5~10)보다 훨씬 큰 수라 `w`의 기울기가 매우 큽니다. 학습률 0.1을 곱한 갱신 크기가 너무 커서 `w`가 최솟값을 크게 지나치고, 이 일이 갱신마다 반복됐습니다.
- 수렴하는 학습률의 범위는 모델뿐 아니라 입력의 크기에 따라서도 달라집니다. 입력을 표준화하면 본실습과 비슷한 학습률을 그대로 쓸 수 있습니다.

</details>

## 9. 핵심 정리

- **학습:** 예측, 손실 계산, 기울기 계산, 파라미터 갱신의 반복입니다. 신경망도 LLM도 같은 반복으로 학습합니다.
- **손실함수:** MSE(`nn.MSELoss`)는 큰 오차를 제곱으로 크게 벌해 이상치에 민감하고, MAE(`nn.L1Loss`)는 오차에 비례해 벌합니다. 손실을 고르기 전에 이상치의 정체부터 확인합니다.
- **기울기와 autograd:** 기울기는 파라미터를 조금 바꿀 때 손실이 늘어나는 방향과 크기입니다. `requires_grad=True`로 만든 파라미터에서 `loss.backward()`를 부르면 `.grad`에 기울기가 담깁니다.
- **역전파:** 손실에서 입력 쪽으로 체인룰을 적용해 단계별 기울기를 곱해 가는 방법이고, `loss.backward()`가 하는 일입니다.
- **경사하강:** `새 파라미터 = 지금 파라미터 - 학습률 × 기울기`를 반복합니다. PyTorch에서는 `optimizer.zero_grad()` → 손실 → `loss.backward()` → `optimizer.step()`입니다. 기울기는 쌓이므로 갱신마다 비웁니다.
- **학습률:** 너무 작으면 느리고, 너무 크면 최솟값을 지나쳐 발산합니다. 입력 크기가 제각각이면 학습률 하나로 맞출 수 없으므로 입력을 표준화합니다. 후보는 10배 간격으로 시도하고, 수렴한 후보 가운데 발산 경계에서 먼 값을 고릅니다.
- **nn.Linear:** `nn.Linear(1, 1)`의 `weight`, `bias`가 직선의 `w`, `b`이고, 뉴런 하나입니다.

교안 02에서는 `nn.Linear`를 여러 층으로 쌓은 신경망(DNN)으로 손글씨 숫자를 분류하고, 미니배치·옵티마이저·학습률 스케줄러를 학습 루프에 더합니다.